# 07. Length Sensitivity

This notebook stratifies Phase 1 performance by short, medium, and long verdicts.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from notebooks.notebook_utils import *

set_plot_style()
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

phase1_raw, _ = load_phase1_results()
phase1_scored = phase1_raw[phase1_raw['faithfulness'].notna()].copy()
phase1_scored['effective_input_tokens'] = phase1_scored['gen_input_tokens'].fillna(phase1_scored['input_tokens'])
sample = load_sample()[['file_id', 'stratum']].rename(columns={'file_id': 'verdict_id'})
phase1_scored = phase1_scored.merge(sample, on='verdict_id', how='left', suffixes=('', '_sample'))
phase1_scored['stratum_final'] = phase1_scored['stratum_sample'].fillna(phase1_scored['stratum'])
phase1_scored[['architecture', 'verdict_id', 'stratum_final', 'faithfulness']].head()

## Faithfulness by stratum

In [ ]:
length_summary = phase1_scored.groupby(['architecture', 'stratum_final'])[['faithfulness', 'effective_input_tokens', 'total_cost_usd']].mean().reset_index()
markdown_df(length_summary)

In [ ]:
sns.barplot(data=length_summary, x='stratum_final', y='faithfulness', hue='architecture', order=['short', 'medium', 'long'])
plt.title('Phase 1 faithfulness by document-length stratum')
plt.tight_layout()

## Cost explosion by stratum

In [ ]:
sns.barplot(data=length_summary, x='stratum_final', y='effective_input_tokens', hue='architecture', order=['short', 'medium', 'long'])
plt.title('Mean input tokens by stratum')
plt.tight_layout()

## Kruskal-Wallis tests within architecture

In [ ]:
rows = []
for architecture, group in phase1_scored.groupby('architecture'):
    values = [bucket['faithfulness'].to_numpy() for _, bucket in group.groupby('stratum_final') if len(bucket) > 0]
    if len(values) >= 2:
        stat, p = stats.kruskal(*values)
        rows.append({'architecture': architecture, 'kruskal_stat': stat, 'pvalue': p})
markdown_df(pd.DataFrame(rows))